# 31 · 72-hour baseline suite on exact frozen support

No neural training. This notebook fits only one simple latest-slot SHARP logistic on the frozen training role, freezes its alarm threshold on model-validation, and compares it with training climatology and the already-frozen SHARP/AIA/fusion probabilities.

Primary evaluation roles are policy-validation, retrospective Cycle 25, and supplementary 2026. Active-region cluster bootstrap intervals use paired resamples across models. Policy remains unchanged.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss

ROOT=Path("/home/abmoses2000")
MASTER=ROOT/"graybox_aia72_master_predictions_v2_20261005"/"graybox_aia72_master_predictions.csv.gz"
LEDGER=ROOT/"sharp72_predictions_frozen_20261002.csv.gz"
SHARP=ROOT/"graybox_phase_h_recovered_input_20261006"/"sharp.npy"
ALARM=ROOT/"graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005"/"alarm_thresholds.csv"
OUT=ROOT/"graybox_aia72_baseline_suite_v2_20261006"

EXPECTED={
 "master":"17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46",
 "ledger":"7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b",
 "sharp":"c9921be7b7fe381b43436a0e927d15d84de83e15376025ae0d8a79a6ef917639",
}
FEATURES=["MEANGBZ","MEANGAM","MEANGBT","MEANGBH","MEANJZD","TOTUSJZ","MEANALP","MEANJZH",
          "ABSNJZH","SAVNCPP","MEANSHR","SHRGT45","R_VALUE","USFLUX","TOTPOT","TOTUSJH"]
EVAL_ROLES=["policy_validation","retrospective_cycle25","supplementary_2026"]
N_BOOT=2000
SEED=720031

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(4*1024*1024),b""): h.update(b)
    return h.hexdigest()

for p in [MASTER,LEDGER,SHARP,ALARM]:
    assert p.is_file(), f"Missing frozen input: {p}"
assert sha256(MASTER)==EXPECTED["master"]
assert sha256(LEDGER)==EXPECTED["ledger"]
assert sha256(SHARP)==EXPECTED["sharp"]
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

master=pd.read_csv(MASTER,low_memory=False)
ledger=pd.read_csv(LEDGER,low_memory=False).sort_values("tensor_row",kind="stable").reset_index(drop=True)
X=np.load(SHARP,mmap_mode="r",allow_pickle=False)
alarm=pd.read_csv(ALARM)

assert X.shape==(113433,3,16)
assert len(ledger)==len(X)
assert np.array_equal(ledger["tensor_row"].to_numpy(int),np.arange(len(X)))
assert int((ledger["role"].eq("train") & ledger["label"].isin([0,1])).sum())==25586
assert int(ledger.loc[ledger["role"].eq("train"),"label"].sum())==1476
assert len(master)==71010

for d in [master,ledger]:
    d["forecast_case_id"]=d["forecast_case_id"].astype(str)

print("Verified exact frozen inputs.")
print("master roles:",master["role"].value_counts().to_dict())
print("ledger roles:",ledger["role"].value_counts().to_dict())


In [ ]:
# Build training features from the exact tensor and authoritative frozen ledger.
train_mask=ledger["role"].eq("train").to_numpy() & ledger["label"].isin([0,1]).to_numpy()
X_latest=np.asarray(X[:,2,:],dtype=float)  # latest available history slot = t-96 min
y_all=ledger["label"].to_numpy(int)

train_X=X_latest[train_mask]
train_y=y_all[train_mask]
train_prev=float(train_y.mean())

# Exact non-train multimodal support: AIA from master; SHARP and tensor_row from frozen ledger.
keep=["forecast_case_id","tensor_row","label","label_known","role","region_component_id","probability_gru_mean"]
ev=master.merge(ledger[keep],on="forecast_case_id",how="inner",suffixes=("_aia","_sharp"),validate="one_to_one")
assert len(ev)==71010

# Frozen role and label are authoritative from the SHARP ledger; verify agreement with master where available.
if "label_aia" in ev.columns:
    assert np.array_equal(pd.to_numeric(ev["label_aia"]).to_numpy(int),pd.to_numeric(ev["label_sharp"]).to_numpy(int))
if "role_aia" in ev.columns:
    assert (ev["role_aia"].astype(str)==ev["role_sharp"].astype(str)).all()

ev["label"]=pd.to_numeric(ev["label_sharp"],errors="raise").astype(int)
ev["role"]=ev["role_sharp"].astype(str)

# region_component_id is not present in every source master prediction artifact.
# Pandas only adds suffixes when the column exists on BOTH sides of the merge.
# Therefore use the authoritative frozen SHARP-ledger field whether it remained
# unsuffixed or became *_sharp, and verify agreement if the master also supplied it.
if "region_component_id_sharp" in ev.columns:
    region_col="region_component_id_sharp"
    if "region_component_id_aia" in ev.columns:
        both=ev["region_component_id_aia"].notna() & ev["region_component_id_sharp"].notna()
        assert (ev.loc[both,"region_component_id_aia"].astype(str)
                ==ev.loc[both,"region_component_id_sharp"].astype(str)).all(), \
               "Master and frozen ledger active-region IDs disagree"
elif "region_component_id" in ev.columns:
    region_col="region_component_id"
else:
    raise KeyError(f"No active-region block column after merge. Columns: {list(ev.columns)}")

assert ev[region_col].notna().all(), "Missing active-region block IDs"
ev["region_component_id"]=ev[region_col].astype(str)

# AIA master raw probability was frozen as the accepted branch probability.
aia_col="probability" if "probability" in ev.columns else "probability_aia"
assert aia_col in ev.columns, list(ev.columns)
ev["p_aia"]=pd.to_numeric(ev[aia_col],errors="raise").astype(float)
ev["p_sharp"]=pd.to_numeric(ev["probability_gru_mean"],errors="raise").astype(float)
ev["p_fusion"]=0.5*ev["p_sharp"]+0.5*ev["p_aia"]
ev["p_climatology"]=train_prev

rows=ev["tensor_row"].to_numpy(int)
ev_latest=X_latest[rows]
assert np.isfinite(ev_latest).all()
assert np.isfinite(ev[["p_aia","p_sharp","p_fusion"]].to_numpy()).all()

print("training prevalence:",train_prev)
print("evaluation rows:",len(ev))


In [ ]:
def confusion_metrics(y,p,t):
    y=np.asarray(y,int); p=np.asarray(p,float)
    pred=(p>=float(t)).astype(int)
    tp=int(((y==1)&(pred==1)).sum()); fn=int(((y==1)&(pred==0)).sum())
    tn=int(((y==0)&(pred==0)).sum()); fp=int(((y==0)&(pred==1)).sum())
    rec=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    tss=rec+spec-1 if np.isfinite(rec) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return {"tss":float(tss),"hss":float(hss),"recall":float(rec),"specificity":float(spec)}

def choose_threshold(y,p):
    cand=np.unique(np.concatenate(([0.0],np.asarray(p,float),[1.0])))
    rows=[]
    for t in cand:
        m=confusion_metrics(y,p,t)
        rows.append({"threshold":float(t),**m})
    q=pd.DataFrame(rows)
    return q.sort_values(["tss","hss","recall","threshold"],
                         ascending=[False,False,False,True],kind="mergesort").iloc[0].to_dict()

def score(y,p,p_ref,threshold):
    y=np.asarray(y,int); p=np.asarray(p,float); p_ref=np.asarray(p_ref,float)
    bs=float(brier_score_loss(y,p))
    bs_ref=float(brier_score_loss(y,p_ref))
    return {
      "cases":int(len(y)),
      "positives":int(y.sum()),
      "prevalence":float(y.mean()),
      "threshold":float(threshold),
      "tss":confusion_metrics(y,p,threshold)["tss"],
      "tss_0.5":confusion_metrics(y,p,0.5)["tss"],
      "average_precision":float(average_precision_score(y,p)),
      "brier":bs,
      "brier_skill_vs_train_climatology":float(1-bs/bs_ref) if bs_ref>0 else np.nan,
    }


In [ ]:
# Fit exactly one simple baseline on frozen training rows only.
pipe=make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=1.0,solver="lbfgs",max_iter=5000),
)
pipe.fit(train_X,train_y)
ev["p_logistic_latest_sharp"]=pipe.predict_proba(ev_latest)[:,1]

# Select the simple logistic alarm threshold only on model_validation, mirroring the frozen branch rule.
mv=ev[ev["role"].eq("model_validation")].copy()
assert len(mv)==3905
mv_idx=mv.index.to_numpy()
logit_thresh=choose_threshold(mv["label"].to_numpy(int),ev.loc[mv_idx,"p_logistic_latest_sharp"].to_numpy(float))

# Existing branch thresholds are already frozen from model_validation.
alarm_map={str(r.branch):float(r.threshold) for r in alarm.itertuples()}
for b in ["sharp","aia","sharp_aia"]:
    assert b in alarm_map, alarm_map

thresholds={
 "climatology":0.5,
 "logistic_latest_sharp":float(logit_thresh["threshold"]),
 "frozen_sharp":alarm_map["sharp"],
 "frozen_aia":alarm_map["aia"],
 "frozen_fusion":alarm_map["sharp_aia"],
}
models={
 "climatology":"p_climatology",
 "logistic_latest_sharp":"p_logistic_latest_sharp",
 "frozen_sharp":"p_sharp",
 "frozen_aia":"p_aia",
 "frozen_fusion":"p_fusion",
}

coef=pd.DataFrame({
 "feature":FEATURES,
 "standardized_logistic_coefficient":pipe.named_steps["logisticregression"].coef_[0],
})
coef.to_csv(OUT/"logistic_latest_sharp_coefficients.csv",index=False)
pd.DataFrame([{"model":k,"threshold":v} for k,v in thresholds.items()]).to_csv(OUT/"baseline_thresholds.csv",index=False)

print("logistic model-validation threshold:",logit_thresh)
print("thresholds:",thresholds)


In [ ]:
# Point metrics on true evaluation roles only.
point=[]
for role in EVAL_ROLES:
    part=ev[ev["role"].eq(role)].copy()
    assert len(part)>0
    y=part["label"].to_numpy(int)
    pref=np.full(len(part),train_prev,dtype=float)
    for name,col in models.items():
        row={"role":role,"model":name}
        row.update(score(y,part[col].to_numpy(float),pref,thresholds[name]))
        point.append(row)
point=pd.DataFrame(point)
point.to_csv(OUT/"baseline_metrics.csv",index=False)
print("===== POINT METRICS =====")
print(point.to_string(index=False))


In [ ]:
# Paired active-region cluster bootstrap: identical sampled ARs for all models within a role.
rng=np.random.default_rng(SEED)
boot_rows=[]

for role in EVAL_ROLES:
    part=ev[ev["role"].eq(role)].reset_index(drop=True)
    blocks=part["region_component_id"].astype(str).to_numpy()
    assert pd.notna(part["region_component_id"]).all()
    uniq=pd.unique(blocks)
    index_map={b:np.flatnonzero(blocks==b) for b in uniq}
    y=part["label"].to_numpy(int)
    pref=np.full(len(part),train_prev,dtype=float)
    probs={name:part[col].to_numpy(float) for name,col in models.items()}

    accepted=0
    for rep in range(N_BOOT):
        draw=rng.choice(uniq,size=len(uniq),replace=True)
        idx=np.concatenate([index_map[b] for b in draw])
        yy=y[idx]
        if len(np.unique(yy))<2:
            continue
        for name,p in probs.items():
            m=score(yy,p[idx],pref[idx],thresholds[name])
            boot_rows.append({"role":role,"replicate":rep,"model":name,
                              "tss":m["tss"],"average_precision":m["average_precision"],
                              "brier":m["brier"],
                              "brier_skill_vs_train_climatology":m["brier_skill_vs_train_climatology"]})
        accepted+=1
    print(role,"accepted bootstrap draws",accepted,"of",N_BOOT)

boot=pd.DataFrame(boot_rows)
boot.to_csv(OUT/"baseline_block_bootstrap_draws.csv.gz",index=False,
            compression={"method":"gzip","mtime":0})

metrics=["tss","average_precision","brier","brier_skill_vs_train_climatology"]
ci=[]
for (role,model),g in boot.groupby(["role","model"],sort=False):
    for metric in metrics:
        a=g[metric].dropna().to_numpy(float)
        ci.append({"role":role,"model":model,"metric":metric,"n_boot":len(a),
                   "boot_median":float(np.median(a)),
                   "ci95_low":float(np.quantile(a,.025)),
                   "ci95_high":float(np.quantile(a,.975))})
ci=pd.DataFrame(ci)
ci.to_csv(OUT/"baseline_block_bootstrap_ci.csv",index=False)

# Paired model deltas from the exact same cluster-bootstrap replicate.
comparisons=[
 ("logistic_latest_sharp","climatology"),
 ("frozen_sharp","climatology"),
 ("frozen_aia","climatology"),
 ("frozen_fusion","climatology"),
 ("frozen_sharp","logistic_latest_sharp"),
 ("frozen_fusion","logistic_latest_sharp"),
 ("frozen_fusion","frozen_sharp"),
]
delta_rows=[]
for role in EVAL_ROLES:
    z=boot[boot.role.eq(role)]
    for a,b in comparisons:
        ga=z[z.model.eq(a)].set_index("replicate")
        gb=z[z.model.eq(b)].set_index("replicate")
        common=ga.index.intersection(gb.index)
        for metric in metrics:
            d=ga.loc[common,metric].to_numpy(float)-gb.loc[common,metric].to_numpy(float)
            delta_rows.append({
              "role":role,"model_a":a,"model_b":b,"metric":metric,
              "delta_definition":"model_a_minus_model_b",
              "better_direction":"higher" if metric!="brier" else "lower",
              "n_boot":int(len(d)),
              "median_delta":float(np.median(d)),
              "ci95_low":float(np.quantile(d,.025)),
              "ci95_high":float(np.quantile(d,.975)),
              "prob_delta_gt_0":float(np.mean(d>0)),
            })
deltas=pd.DataFrame(delta_rows)
deltas.to_csv(OUT/"baseline_paired_bootstrap_deltas.csv",index=False)

print("===== BLOCK BOOTSTRAP CI =====")
print(ci.to_string(index=False))
print("\n===== PAIRED DELTAS =====")
print(deltas.to_string(index=False))


In [ ]:
receipt={
 "status":"72H_BASELINE_SUITE_COMPLETE",
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "policy_changed":False,
 "neural_models_refit":False,
 "simple_model_fit":"one L2 logistic on 16 latest-slot SHARP features; train role only",
 "training_rows":int(train_mask.sum()),
 "training_positives":int(train_y.sum()),
 "training_prevalence":train_prev,
 "logistic_threshold_selected_on":"model_validation",
 "evaluation_roles":EVAL_ROLES,
 "bootstrap":{"unit":"region_component_id","draws_requested":N_BOOT,"seed":SEED,"paired_across_models":True},
 "thresholds":thresholds,
 "source_sha256":{
   "master":sha256(MASTER),
   "sharp_ledger":sha256(LEDGER),
   "sharp_tensor":sha256(SHARP),
   "alarm_thresholds":sha256(ALARM),
 },
 "claim_boundary":"Manuscript baseline comparison only; no Gray-Box policy or Phase-H monitor change.",
}
(OUT/"baseline_receipt.json").write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))
